In [6]:
import pandas as pd

# Charger les 2 datasets
results = pd.read_csv('../data/raw/results.csv')
rankings = pd.read_csv('../data/raw/fifa_ranking-2024-06-20.csv')

# Premier coup d'oeil
print("RESULTS:")
print(results.head())
print(results.shape)

print("\nRANKINGS:")
print(rankings.head())
print(rankings.shape)

RESULTS:
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49509, 9)

RANKINGS:
    rank       country_full country_abrv  total_points  previous_points  \
0  140.0  Brunei Darussalam          BRU           2.0              0.0   
1   33.0           Portugal          POR          38.0              0.0   
2   32.0             Zambia          ZAM          38.0              0.0   
3   31.0             Greece          GRE    

In [7]:
check = pd.read_csv('../data/raw/fifa_ranking-2024-06-20.csv')
print(check.columns.tolist())
print(check['rank_date'].unique() if 'rank_date' in check.columns else "pas de colonne rank_date")
print(check.shape)

['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']
<StringArray>
['1992-12-31', '1993-08-08', '1993-09-23', '1993-10-22', '1993-11-19',
 '1993-12-23', '1994-02-15', '1994-03-15', '1994-04-19', '1994-05-17',
 ...
 '2023-04-06', '2023-06-29', '2023-07-20', '2023-09-21', '2023-10-26',
 '2023-11-30', '2023-12-21', '2024-02-15', '2024-04-04', '2024-06-20']
Length: 333, dtype: str
(67472, 8)


In [8]:
print(results.columns.tolist())
print(results.head())
print(results.shape)

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3  1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4  1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49509, 9)


## DATA CLEANING

In [9]:
results['date'] = pd.to_datetime(results['date'])
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])

In [10]:
results = results.sort_values('date').reset_index(drop=True)
rankings = rankings.sort_values('rank_date').reset_index(drop=True)

In [12]:
teams_results = set(results['home_team'].unique()) | set(results['away_team'].unique())
teams_rankings = set(rankings['country_full'].unique())

# Noms présents dans results mais absents de rankings
missing = teams_results - teams_rankings
print(f"Nombre d'équipes sans correspondance: {len(missing)}")
# Filtrer les NaN avant de trier
missing_clean = {x for x in missing if isinstance(x, str)}
print(f"Nombre d'équipes sans correspondance: {len(missing_clean)}")
print(sorted(missing_clean))

# Vérifier s'il y avait bien des NaN
nan_count = len(missing) - len(missing_clean)
print(f"\nValeurs NaN trouvées: {nan_count}")

Nombre d'équipes sans correspondance: 143
Nombre d'équipes sans correspondance: 142
['Abkhazia', 'Alderney', 'Ambazonia', 'Andalusia', 'Arameans Suryoye', 'Artsakh', 'Asturias', 'Aymara', 'Barawa', 'Basque Country', 'Biafra', 'Bonaire', 'Brittany', 'Brunei', 'Canary Islands', 'Cape Verde', 'Cascadia', 'Catalonia', 'Central Spain', 'Chagos Islands', 'Chameria', 'Chechnya', 'China', 'Cilento', 'Corsica', 'County of Nice', 'Crimea', 'Curaçao', 'Czech Republic', 'DR Congo', 'Darfur', 'Donetsk PR', 'Délvidék', 'East Turkestan', 'Elba Island', 'Ellan Vannin', 'Falkland Islands', 'Felvidék', 'Franconia', 'French Guiana', 'Frøya', 'Galicia', 'Gambia', 'German DR', 'Gotland', 'Gozo', 'Greenland', 'Guadeloupe', 'Guernsey', 'Găgăuzia', 'Hitra', 'Hmong', 'Iran', 'Iraqi Kurdistan', 'Isle of Man', 'Isle of Wight', 'Ivory Coast', 'Jersey', 'Kabylia', 'Kernow', 'Kiribati', 'Kurdistan', 'Kyrgyzstan', 'Kárpátalja', 'Luhansk PR', 'Madrid', 'Manchukuo', 'Mapuche', 'Marshall Islands', 'Martinique', 'Matabe

In [17]:
name_mapping = {
    'United States': 'USA',
    'South Korea': 'Korea Republic',
    'North Korea': 'Korea DPR',
    "Ivory Coast": "Côte d'Ivoire",
    'Czech Republic': 'Czechia',
    'DR Congo': 'Congo DR',  # corrigé
    'Iran': 'IR Iran',
    'Gambia': 'The Gambia',
    'Cape Verde': 'Cabo Verde',
    'Brunei': 'Brunei Darussalam',
    'Taiwan': 'Chinese Taipei',
    'China': 'China PR',
    'Saint Kitts and Nevis': 'St Kitts and Nevis',
    'Saint Lucia': 'St Lucia',
    'Saint Vincent and the Grenadines': 'St Vincent and the Grenadines',  # corrigé
    'Kyrgyzstan': 'Kyrgyz Republic',
}

# Ré-appliquer le mapping corrigé
results['home_team'] = results['home_team'].replace(name_mapping)
results['away_team'] = results['away_team'].replace(name_mapping)

In [18]:
teams_results_v3 = set(results['home_team'].unique()) | set(results['away_team'].unique())
still_missing_v2 = {x for x in (teams_results_v3 - teams_rankings) if isinstance(x, str)}
print(f"Restant après mapping corrigé: {len(still_missing_v2)}")

Restant après mapping corrigé: 127


In [16]:
print('Spain' in teams_rankings, 'Argentina' in teams_rankings)
print('Spain' in teams_results_v2, 'Argentina' in teams_results_v2)

True True
True True


In [19]:
# Sécurité : reconvertir les dates si pas encore fait
results['date'] = pd.to_datetime(results['date'])
rankings['rank_date'] = pd.to_datetime(rankings['rank_date'])

# Trier (obligatoire pour merge_asof)
results = results.sort_values('date').reset_index(drop=True)
rankings = rankings.sort_values('rank_date').reset_index(drop=True)

# Merge pour l'équipe à domicile (home_team)
df = pd.merge_asof(
    results,
    rankings[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date', right_on='rank_date',
    left_by='home_team', right_by='country_full',
    direction='backward'
).rename(columns={'rank': 'home_rank', 'total_points': 'home_points'})

df = df.drop(columns=['rank_date', 'country_full'])

# Merge pour l'équipe à l'extérieur (away_team)
df = pd.merge_asof(
    df,
    rankings[['rank_date', 'country_full', 'rank', 'total_points']],
    left_on='date', right_on='rank_date',
    left_by='away_team', right_by='country_full',
    direction='backward'
).rename(columns={'rank': 'away_rank', 'total_points': 'away_points'})

df = df.drop(columns=['rank_date', 'country_full'])

print(df.shape)
print(df.head())
print(df.isnull().sum())

(49509, 13)
        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   
3 1875-03-06   England  Scotland         2.0         2.0   Friendly   London   
4 1876-03-04  Scotland   England         3.0         0.0   Friendly  Glasgow   

    country  neutral  home_rank  home_points  away_rank  away_points  
0  Scotland    False        NaN          NaN        NaN          NaN  
1   England    False        NaN          NaN        NaN          NaN  
2  Scotland    False        NaN          NaN        NaN          NaN  
3   England    False        NaN          NaN        NaN          NaN  
4  Scotland    False        NaN          NaN        NaN          NaN  
date               0
home_team          2
away_team          2
home_score        

In [20]:
# Vérifier les lignes avec équipes ou scores manquants
print(df[df['home_team'].isnull() | df['away_team'].isnull()])
print(df[df['home_score'].isnull() | df['away_score'].isnull()])

            date home_team away_team  home_score  away_score      tournament  \
49507 2026-07-18       NaN       NaN         NaN         NaN  FIFA World Cup   
49508 2026-07-19       NaN       NaN         NaN         NaN  FIFA World Cup   

                  city        country  neutral  home_rank  home_points  \
49507    Miami Gardens  United States     True        NaN          NaN   
49508  East Rutherford  United States     True        NaN          NaN   

       away_rank  away_points  
49507        NaN          NaN  
49508        NaN          NaN  
            date home_team  away_team  home_score  away_score      tournament  \
49505 2026-07-14    France      Spain         NaN         NaN  FIFA World Cup   
49506 2026-07-15   England  Argentina         NaN         NaN  FIFA World Cup   
49507 2026-07-18       NaN        NaN         NaN         NaN  FIFA World Cup   
49508 2026-07-19       NaN        NaN         NaN         NaN  FIFA World Cup   

                  city        coun

In [21]:
# Dropper les lignes complètement vides (les 2 placeholders sans équipes)
df = df.dropna(subset=['home_team', 'away_team'])

print(f"Shape après suppression des placeholders vides: {df.shape}")

Shape après suppression des placeholders vides: (49507, 13)


In [22]:
# Garder uniquement les matchs avec ranking FIFA connu ET un score
df_clean = df.dropna(subset=['home_rank', 'away_rank', 'home_score', 'away_score'])

print(f"Avant filtrage: {df.shape}")
print(f"Après filtrage: {df_clean.shape}")
print(df_clean['date'].min(), "->", df_clean['date'].max())

Avant filtrage: (49507, 13)
Après filtrage: (28183, 13)
1993-01-01 00:00:00 -> 2026-07-11 00:00:00


## FEATURE ENGINEERING

In [23]:
# Feature de base : différence de classement et de points FIFA
df_clean = df_clean.copy()  # éviter le warning pandas
df_clean['rank_diff'] = df_clean['away_rank'] - df_clean['home_rank']
df_clean['points_diff'] = df_clean['home_points'] - df_clean['away_points']

# Target : qui a gagné ? (1 = victoire home, 0 = victoire away, on garde de côté les nuls pour l'instant)
df_clean['home_win'] = (df_clean['home_score'] > df_clean['away_score']).astype(int)
df_clean['away_win'] = (df_clean['away_score'] > df_clean['home_score']).astype(int)
df_clean['draw'] = (df_clean['home_score'] == df_clean['away_score']).astype(int)

print(df_clean[['home_team', 'away_team', 'rank_diff', 'points_diff', 'home_win', 'away_win', 'draw']].head(10))
print("\nRépartition:")
print(f"Victoires home: {df_clean['home_win'].sum()}")
print(f"Victoires away: {df_clean['away_win'].sum()}")
print(f"Nuls: {df_clean['draw'].sum()}")

          home_team     away_team  rank_diff  points_diff  home_win  away_win  \
18708         Ghana          Mali       30.0         12.0         0         0   
18709         Gabon  Burkina Faso       42.0         16.0         0         0   
18710        Kuwait       Lebanon       90.0         21.0         1         0   
18711  Burkina Faso          Mali      -28.0        -11.0         1         0   
18712         Gabon         Ghana      -16.0         -7.0         0         1   
18713        Uganda      Tanzania      -12.0         -3.0         1         0   
18714        Angola      Zimbabwe      -48.0        -17.0         0         0   
18715      Botswana  South Africa      -15.0         -3.0         0         1   
18717       Senegal       Algeria      -21.0        -12.0         0         1   
18718       Tunisia      Bulgaria      -12.0         -6.0         1         0   

       draw  
18708     1  
18709     1  
18710     0  
18711     0  
18712     0  
18713     0  
18714     

In [24]:
# D'abord, on construit un historique "long format" : une ligne par équipe par match
home_df = df_clean[['date', 'home_team', 'home_win', 'draw', 'away_win']].copy()
home_df.columns = ['date', 'team', 'win', 'draw', 'loss']

away_df = df_clean[['date', 'away_team', 'away_win', 'draw', 'home_win']].copy()
away_df.columns = ['date', 'team', 'win', 'draw', 'loss']

team_history = pd.concat([home_df, away_df]).sort_values(['team', 'date']).reset_index(drop=True)

print(team_history.head(10))
print(team_history.shape)

        date         team  win  draw  loss
0 2003-03-16  Afghanistan    1     0     0
1 2003-03-18  Afghanistan    0     0     1
2 2003-11-19  Afghanistan    0     0     1
3 2003-11-23  Afghanistan    0     0     1
4 2005-11-09  Afghanistan    0     0     1
5 2005-12-07  Afghanistan    0     0     1
6 2005-12-09  Afghanistan    0     0     1
7 2005-12-11  Afghanistan    1     0     0
8 2006-04-03  Afghanistan    0     1     0
9 2006-04-05  Afghanistan    0     1     0
(56366, 5)


## ROLLING AVERAAGE

In [25]:
# Pour chaque équipe, calculer le nombre de victoires sur les 5 derniers matchs (avant le match actuel)
team_history['win_last5'] = (
    team_history.groupby('team')['win']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

team_history['loss_last5'] = (
    team_history.groupby('team')['loss']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
)

print(team_history[team_history['team'] == 'Afghanistan'].head(12))

         date         team  win  draw  loss  win_last5  loss_last5
0  2003-03-16  Afghanistan    1     0     0        NaN         NaN
1  2003-03-18  Afghanistan    0     0     1        1.0         0.0
2  2003-11-19  Afghanistan    0     0     1        1.0         1.0
3  2003-11-23  Afghanistan    0     0     1        1.0         2.0
4  2005-11-09  Afghanistan    0     0     1        1.0         3.0
5  2005-12-07  Afghanistan    0     0     1        1.0         4.0
6  2005-12-09  Afghanistan    0     0     1        0.0         5.0
7  2005-12-11  Afghanistan    1     0     0        0.0         5.0
8  2006-04-03  Afghanistan    0     1     0        1.0         4.0
9  2006-04-05  Afghanistan    0     1     0        1.0         3.0
10 2007-10-08  Afghanistan    0     0     1        1.0         2.0
11 2007-10-26  Afghanistan    0     0     1        1.0         2.0


In [26]:
# Merge pour l'équipe home
df_clean = df_clean.merge(
    team_history[['date', 'team', 'win_last5', 'loss_last5']],
    left_on=['date', 'home_team'], right_on=['date', 'team'],
    how='left'
).rename(columns={'win_last5': 'home_form_win', 'loss_last5': 'home_form_loss'}).drop(columns=['team'])

# Merge pour l'équipe away
df_clean = df_clean.merge(
    team_history[['date', 'team', 'win_last5', 'loss_last5']],
    left_on=['date', 'away_team'], right_on=['date', 'team'],
    how='left'
).rename(columns={'win_last5': 'away_form_win', 'loss_last5': 'away_form_loss'}).drop(columns=['team'])

print(df_clean.shape)
print(df_clean[['home_team', 'away_team', 'home_form_win', 'home_form_loss', 'away_form_win', 'away_form_loss']].head(10))
print(df_clean.isnull().sum())

(28205, 22)
      home_team     away_team  home_form_win  home_form_loss  away_form_win  \
0         Ghana          Mali            NaN             NaN            NaN   
1         Gabon  Burkina Faso            NaN             NaN            NaN   
2        Kuwait       Lebanon            NaN             NaN            NaN   
3  Burkina Faso          Mali            0.0             0.0            0.0   
4         Gabon         Ghana            0.0             0.0            0.0   
5        Uganda      Tanzania            NaN             NaN            NaN   
6        Angola      Zimbabwe            NaN             NaN            NaN   
7      Botswana  South Africa            NaN             NaN            NaN   
8       Senegal       Algeria            NaN             NaN            NaN   
9       Tunisia      Bulgaria            NaN             NaN            NaN   

   away_form_loss  
0             NaN  
1             NaN  
2             NaN  
3             0.0  
4             0.0 

In [27]:
# Vérifier s'il y a des vrais doublons de lignes après le merge
duplicates = df_clean[df_clean.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)]
print(duplicates.shape)
print(duplicates[['date', 'home_team', 'away_team', 'home_score', 'away_score']].head(20))

(40, 22)
            date    home_team         away_team  home_score  away_score
1285  1995-03-25     Pakistan        Bangladesh         1.0         0.0
1286  1995-03-25     Pakistan        Bangladesh         1.0         0.0
1287  1995-03-25        Nepal        Bangladesh         0.0         2.0
1288  1995-03-25        Nepal        Bangladesh         0.0         2.0
5527  2000-09-03     China PR              Iraq         4.0         1.0
5528  2000-09-03     China PR              Iraq         4.0         1.0
5536  2000-09-03    Indonesia              Iraq         3.0         0.0
5537  2000-09-03    Indonesia              Iraq         3.0         0.0
5790  2001-01-17       Israel        Uzbekistan         2.0         0.0
5791  2001-01-17       Israel        Uzbekistan         2.0         0.0
5793  2001-01-17        Chile        Uzbekistan         2.0         0.0
5794  2001-01-17        Chile        Uzbekistan         2.0         0.0
13793 2010-05-30        Chile  Northern Ireland        

In [28]:
df_clean = df_clean.drop_duplicates(subset=['date', 'home_team', 'away_team', 'home_score', 'away_score'])
print(df_clean.shape)  # devrait revenir à ~28183

(28182, 22)


In [29]:
df_clean[['home_form_win', 'home_form_loss', 'away_form_win', 'away_form_loss']] = (
    df_clean[['home_form_win', 'home_form_loss', 'away_form_win', 'away_form_loss']].fillna(0)
)

print(df_clean.isnull().sum())

date              0
home_team         0
away_team         0
home_score        0
away_score        0
tournament        0
city              0
country           0
neutral           0
home_rank         0
home_points       0
away_rank         0
away_points       0
rank_diff         0
points_diff       0
home_win          0
away_win          0
draw              0
home_form_win     0
home_form_loss    0
away_form_win     0
away_form_loss    0
dtype: int64
